In [4]:
import pandas as pd

idx = 81
vocal_audio_dir = f'../data/clean/audio/vocals/{idx}.mp3'
raw_audio_dir = f'../data/raw/audio/{idx}.mp3'

segment_index_file = "../models/predict_syllables/segment_break.csv" # f'../data/clean/syllables/segment_index.csv'
segment_index = pd.read_csv(segment_index_file)
segment_index = segment_index[segment_index['file'] == idx]
segment_index.head()

,file,start,end,pred
0,81,0,12400,<silence>
1,81,12400,12580,<silence>
2,81,12580,12720,<silence>
3,81,12720,12920,<silence>
4,81,12920,13060,い


In [5]:
def hiragana_to_romaji(token):
    token_set =  {
        # Basic vowels
        'あ': 'a',    'い': 'i',    'う': 'u',    'え': 'e',    'お': 'o',

        # Basic consonant + vowel
        'か': 'ka',   'き': 'ki',   'く': 'ku',   'け': 'ke',   'こ': 'ko',
        'さ': 'sa',   'し': 'shi',  'す': 'su',   'せ': 'se',   'そ': 'so',
        'た': 'ta',   'ち': 'chi',  'つ': 'tsu',  'て': 'te',   'と': 'to',
        'な': 'na',   'に': 'ni',   'ぬ': 'nu',   'ね': 'ne',   'の': 'no',
        'は': 'ha',   'ひ': 'hi',   'ふ': 'fu',   'へ': 'he',   'ほ': 'ho',
        'ま': 'ma',   'み': 'mi',   'む': 'mu',   'め': 'me',   'も': 'mo',
        'や': 'ya',   'ゆ': 'yu',   'よ': 'yo',
        'ら': 'ra',   'り': 'ri',   'る': 'ru',   'れ': 're',   'ろ': 'ro',
        'わ': 'wa',   'を': 'wo',   'ん': 'n',

        # Dakuten (voiced) sounds
        'が': 'ga',   'ぎ': 'gi',   'ぐ': 'gu',   'げ': 'ge',   'ご': 'go',
        'ざ': 'za',   'じ': 'ji',   'ず': 'zu',   'ぜ': 'ze',   'ぞ': 'zo',
        'だ': 'da',   'ぢ': 'ji',   'づ': 'zu',   'で': 'de',   'ど': 'do',
        'ば': 'ba',   'び': 'bi',   'ぶ': 'bu',   'べ': 'be',   'ぼ': 'bo',

        # Handakuten (semi-voiced) sounds
        'ぱ': 'pa',   'ぴ': 'pi',   'ぷ': 'pu',   'ぺ': 'pe',   'ぽ': 'po',

        # Yōon (combined sounds)
        'きゃ': 'kya', 'きゅ': 'kyu', 'きょ': 'kyo',
        'しゃ': 'sha', 'しゅ': 'shu', 'しょ': 'sho',
        'ちゃ': 'cha', 'ちゅ': 'chu', 'ちょ': 'cho',
        'にゃ': 'nya', 'にゅ': 'nyu', 'にょ': 'nyo',
        'ひゃ': 'hya', 'ひゅ': 'hyu', 'ひょ': 'hyo',
        'みゃ': 'mya', 'みゅ': 'myu', 'みょ': 'myo',
        'りゃ': 'rya', 'りゅ': 'ryu', 'りょ': 'ryo',
        'ぎゃ': 'gya', 'ぎゅ': 'gyu', 'ぎょ': 'gyo',
        'じゃ': 'ja',  'じゅ': 'ju',  'じょ': 'jo',
        'びゃ': 'bya', 'びゅ': 'byu', 'びょ': 'byo',
        'ぴゃ': 'pya', 'ぴゅ': 'pyu', 'ぴょ': 'pyo',

        # Additional special sounds
        'でぃ': 'di',
        'ふぁ': 'fa',  'ふぃ': 'fi',  'ふぇ': 'fe',  'ふぉ': 'fo',

        # Special tokens
        '<silence>': '[silence]',
        '<bre>': '[breath]',
        '<echo>': '[echo]'
    }
    
    return token_set.get(token, token)

In [11]:
def ms_to_srt_time(ms):
    """Convert milliseconds to SRT timestamp format"""
    seconds = ms // 1000
    ms = ms % 1000
    minutes = seconds // 60
    seconds = seconds % 60
    hours = minutes // 60
    minutes = minutes % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{ms:03d}"

def create_srt(segment_index):
    srt_entries = []
    
    has_token_column = False
    
    if 'token' in segment_index.columns:
        has_token_column = True
        segment_index['token'] = segment_index['token'].map(hiragana_to_romaji)
    
    segment_index['pred'] = segment_index['pred'].map(hiragana_to_romaji)

    for idx, row in segment_index.iterrows():
        counter = idx - segment_index.index[0] + 1
        start_time = ms_to_srt_time(row['start'])
        end_time = ms_to_srt_time(row['end'])

        # Color coding: green for token, white/red for pred based on match
        token_thing = '' if not has_token_column else row['token'] 
        token = f'\033{token_thing}\033'  # green
        if has_token_column and row['token'] == row['pred']:
            pred = f'{row["pred"]}'  # white
        else:
            pred = f'\033{row["pred"]}\033'  # red

        text = f"{token}\n{pred}"

        srt_entry = f"{counter}\n{start_time} --> {end_time}\n{text}\n"
        srt_entries.append(srt_entry)

    return "\n".join(srt_entries)

# Generate SRT content
srt_content = create_srt(segment_index)

# Write to file
# output_file = f'{idx}.srt'
for i in [81,83,85,89,90,91,92]:
    output_file = f'outputs/{i}.srt'
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(srt_content)


In [8]:
def ms_to_vtt_time(ms):
    """Convert milliseconds to WebVTT timestamp format"""
    seconds = ms / 1000
    minutes = int(seconds // 60)
    seconds = seconds % 60
    hours = int(minutes // 60)
    minutes = minutes % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:06.3f}"

def create_vtt(segment_index):
    # WebVTT header with styling
    vtt_header = """WEBVTT

STYLE
::cue {
    font-size: 200%;
    line-height: 1.5;
    background-color: transparent;
}
::cue(.green) {
    color: lime;
}
::cue(.red) {
    color: red;
}

"""
    vtt_entries = [vtt_header]
    
    has_token_column = False
    if 'token' in segment_index.columns:
        has_token_column = True
        segment_index['token'] = segment_index['token'].map(hiragana_to_romaji)
    segment_index['pred'] = segment_index['pred'].map(hiragana_to_romaji)
    

    for idx, row in segment_index.iterrows():
        start_time = ms_to_vtt_time(row['start'])
        end_time = ms_to_vtt_time(row['end'])
        if has_token_column:
            token = f'<c.green>{row["token"]}</c>'
        else:
            token = ''
        
        if has_token_column:
            if row['token'] == row['pred']:
                pred = f'<c.green>{row["pred"]}</c>'  # default color (white)
            else:
                pred = f'<c.red>{row["pred"]}</c>'
        else:
            pred = f'<c.white>{row["pred"]}</c>'

        text = f"{token}\n{pred}"

        vtt_entry = f"{start_time} --> {end_time}\n{text}\n\n"
        vtt_entries.append(vtt_entry)

    return "".join(vtt_entries)

# Generate VTT content
vtt_content = create_vtt(segment_index)

# Write to file
for i in [81,83,85,89,90,91,92]:
    output_file = f'outputs/{i}.vtt'
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(vtt_content)